In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util, CrossEncoder
import pandas as pd
import torch
import os
from pathlib import Path
from tqdm import tqdm



> Query



In [ ]:
df_q = pd.read_csv("acgs_d.csv")



> Corpus



In [ ]:
companies = [
    "PT AKR Corporindo Tbk",
    "PT Bank CIMB Niaga Tbk",
    "PT Bank Central Asia Tbk",
    "PT Bank Jago Tbk",
    "PT Bank Mandiri Tbk",
    "PT Bank Negara Indonesia Tbk",
    "PT Bank Rakyat Indonesia Tbk",
    "PT Indosat Tbk",
    "PT Telkom Indonesia Tbk",
    "PT Wijaya Karya Tbk",
]

In [ ]:
company_offsets = {
    "PT AKR Corporindo Tbk": 2,
    "PT Bank CIMB Niaga Tbk": -2,
    "PT Bank Central Asia Tbk": 2,
    "PT Bank Jago Tbk": 2,
    "PT Bank Mandiri Tbk": 2,
    "PT Bank Negara Indonesia Tbk": 2,
    "PT Bank Rakyat Indonesia Tbk": 2,
    "PT Indosat Tbk": 2,
    "PT Telkom Indonesia Tbk": 2,
    "PT Wijaya Karya Tbk": 0,
}



> Top-K



In [ ]:
topks = [5, 10, 20]



> Models



In [ ]:
retriever_model = SentenceTransformer("multi-qa-mpnet-base-dot-v1", device="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", device="cuda")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

# Helper Function



> Retriever Encoding



In [ ]:
def encode_text(text):
    emb = retriever_model.encode(
        text,
        show_progress_bar=True,
        convert_to_tensor=True
    )
    return emb



> Querying



In [ ]:
def map_to_page_level(queries, pages, ret_scores):
    page_level_ret_res_lod = []

    for i in range(len(queries)):
        scores_q = ret_scores[i].cpu().numpy()

        for j, score in enumerate(scores_q):
            page_level_ret_res_lod.append({
                "q_idx": i,
                "d_idx": j,
                "page_retrieved": pages[j],
                "score": score
            })

    return pd.DataFrame(page_level_ret_res_lod)

In [ ]:
# def max_aggregate(page_level_ret_res, topk):
#     page_level_ret_res = page_level_ret_res.sort_values(["q_idx", "score"], ascending=[True, False]).reset_index(drop=True)
#     page_level_ret_res = page_level_ret_res.drop_duplicates(subset=["q_idx", "page_retrieved"], keep="first")
#
#     aggregated_ret_res = page_level_ret_res.groupby("q_idx").head(topk).reset_index(drop=True)
#     aggregated_ret_res["rank"] = aggregated_ret_res.groupby("q_idx").cumcount() + 1
#
#     return aggregated_ret_res

In [ ]:
def max_aggregate(page_level_ret_res, topk):

    idx = (
        page_level_ret_res
        .groupby(["q_idx", "page_retrieved"])["score"]
        .idxmax()
    )

    page_level_ret_res = page_level_ret_res.loc[idx].reset_index(drop=True)

    page_level_ret_res = page_level_ret_res.sort_values(
        ["q_idx", "score"],
        ascending=[True, False]
    ).reset_index(drop=True)

    aggregated_ret_res = (
        page_level_ret_res
        .groupby("q_idx")
        .head(topk)
        .reset_index(drop=True)
    )

    aggregated_ret_res["rank"] = (
        aggregated_ret_res.groupby("q_idx").cumcount() + 1
    )

    return aggregated_ret_res

> Evaluation

In [ ]:
def parse_anchor_pages(s, offset=0):
    pages = set()

    if pd.isna(s):
        return pages

    s = str(s).replace('"', '').replace("'", "").strip()
    if s == "":
        return pages

    for part in s.split(";"):
        part = part.strip()
        if part == "":
            continue

        if "-" in part:
            a, b = part.split("-", 1)
            a, b = a.strip(), b.strip()
            if a == "" or b == "":
                continue
            pages.update(range(int(a) + offset, int(b) + offset + 1))
        else:
            pages.add(int(part) + offset)

    return pages

In [ ]:
def merge_parse_df(topk_df, query_df, corpus_df, evaluator_df, offset):
    df_merged = (
        topk_df
        .merge(query_df, left_on='q_idx', right_index=True, how='left')
        .merge(corpus_df, left_on='d_idx', right_index=True, how='left')
        .merge(evaluator_df, left_on='q_idx', right_index=True, how='left')
    )

    df_merged["anchor_set"] = df_merged["anchor_pages"].apply(
        lambda x: parse_anchor_pages(x, offset=offset)
    )

    return df_merged

In [ ]:
def filter_df(df):
    df_final = df[(df["answerable"] == True) & (df["in_scope"] == True)]
    df_final = df_final.copy()
    df_final["is_hit"] = df_final.apply(lambda r: r["page"] in r["anchor_set"], axis=1)

    return df_final

In [ ]:
def generate_score(df, company, topk):

    hit_score = (
        df[df["rank"] <= topk]
        .groupby("qid")["is_hit"]
        .any()
        .mean()
    )

    df_k = df[df["rank"] <= topk].copy()

    hits = df_k[df_k["is_hit"]]

    first_hit_rank = (
        hits
        .sort_values(["qid", "rank"])
        .groupby("qid")["rank"]
        .first()
    )

    reciprocal = 1 / first_hit_rank

    mrr_score = (
        reciprocal
        .reindex(df["qid"].unique(), fill_value=0)
        .mean()
    )

    return {
        "company": company,
        "k": int(topk),
        "hit_at_k": float(hit_score),
        "mrr_at_k": float(mrr_score)
    }

# Main Loop

In [ ]:
queries = df_q["q_text"].tolist()
emb_q = encode_text(queries)

df_ret = []

retriever_eval = []
reranker_eval = []

all_final_ret = []
all_final_rer = []

candidate_pool = 100

for company in tqdm(companies):
    corpus = pd.read_excel(f"{company}.xlsx")
    documents = corpus["d_text"].tolist()
    pages = corpus["page"].tolist()

    evaluator = pd.read_excel(f"{company}_Evaluator.xlsx")

    offset = company_offsets.get(company, 0)

    # retriever encoding
    emb_d = encode_text(documents)

    # querying
    ret_scores = util.dot_score(emb_q, emb_d)

    page_level_ret_res = map_to_page_level(queries, pages, ret_scores)

    page_level_ret_res_top = (
        page_level_ret_res
        .sort_values(["q_idx", "score"], ascending=[True, False])
        .groupby("q_idx")
        .head(candidate_pool)
        .reset_index(drop=True)
    )



    for topk in topks:
        # retriever eval
        aggregated_ret_res = max_aggregate(page_level_ret_res_top, topk)

        merged_df = merge_parse_df(aggregated_ret_res, df_q, corpus, evaluator, offset)

        final_df_ret = filter_df(merged_df)
        final_df_ret["company"] = company #wad

        ret_eval = generate_score(final_df_ret, company, topk)

        retriever_eval.append(ret_eval)

    all_final_ret.append(final_df_ret)

    # reranker encoding
    reranked = []

    sorted_ret_res = page_level_ret_res.sort_values(["q_idx", "score"], ascending=[True, False]).reset_index(drop=True)
    sorted_ret_res = sorted_ret_res.groupby("q_idx").head(candidate_pool).reset_index(drop=True)

    merged_df_rer = merge_parse_df(sorted_ret_res, df_q, corpus, evaluator, offset)

    for idx, df in merged_df_rer.groupby("q_idx"):
        pairs = list(zip(df["q_text"].tolist(), df["d_text"].tolist()))
        rer_scores = reranker_model.predict(pairs)

        df = df.copy()
        df["rerank_score"] = rer_scores

        idx_max = df.groupby("page_retrieved")["rerank_score"].idxmax()
        df_page = df.loc[idx_max].sort_values("rerank_score", ascending=False).reset_index(drop=True)
        df_page["rank"] = df_page.index + 1
        df_page["q_idx"] = idx

        reranked.append(df_page)

    reranked_df = pd.concat(reranked, ignore_index=True)

    final_df_rer = filter_df(reranked_df)
    final_df_rer["company"] = company
    all_final_rer.append(final_df_rer)

    for topk in topks:
        rer_eval = generate_score(final_df_rer, company, topk)
        reranker_eval.append(rer_eval)

    print(f" - {company} done")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:24<03:43, 24.79s/it]

 - PT AKR Corporindo Tbk done


...

 90%|█████████ | 9/10 [04:24<00:29, 29.35s/it]

 - PT Telkom Indonesia Tbk done


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

100%|██████████| 10/10 [04:51<00:00, 29.11s/it]

 - PT Wijaya Karya Tbk done


# Safe

In [ ]:
reranked_df = pd.DataFrame(reranked_df)
retriever_eval = pd.DataFrame(retriever_eval)
reranker_eval = pd.DataFrame(reranker_eval)

combined_ret = pd.concat(all_final_ret, ignore_index=True)
combined_rer = pd.concat(all_final_rer, ignore_index=True)

In [ ]:
reranked_df.to_csv("reranked_df.csv", index=False)
retriever_eval.to_csv("retriever_eval.csv", index=False)
reranker_eval.to_csv("reranker_eval.csv", index=False)
combined_ret.to_csv("all_final_ret.csv", index=False)
combined_rer.to_csv("all_final_rer.csv", index=False)